# 05 · Image embeddings and similarity search

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/05_embeddings.ipynb)

An image encoder turns each image into a vector. Similar images get similar vectors,
which is useful for de-duplication, clustering and search. This notebook uses
BioCLIP 2 (MIT), a model trained on the tree of life.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
fetch("demo_data_det.zip")
fetch("demo_data_base.zip")  # about 95 MB
squirrels = sorted((DATA / "data/data_example/images").rglob("*.jpg"))[:6]
amazon = [DATA / "demo_data/imgs/10050028_1.JPG", DATA / "demo_data/imgs/10050028_2.JPG"]
images = squirrels + amazon

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["bioclip-2-fp16"]:
    spe.download_model(model_id)

In [ ]:
import numpy as np

emb = spe.embed(images, model="bioclip-2-fp16")
print(emb.shape)  # (images, dimensions); vectors are L2-normalized
sim = emb @ emb.T

In [ ]:
import matplotlib.pyplot as plt

labels = [f"squirrel {i}" for i in range(len(squirrels))] + ["amazon 1", "amazon 2"]
plt.figure(figsize=(6, 5))
plt.imshow(sim, vmin=0.5, vmax=1, cmap="viridis")
plt.xticks(range(len(labels)), labels, rotation=60, ha="right")
plt.yticks(range(len(labels)), labels)
plt.colorbar(label="cosine similarity")
plt.title("BioCLIP 2 similarity")

## Nearest neighbours

In [ ]:
query = 0
order = np.argsort(-sim[query])
for i in order[:4]:
    print(f"{sim[query, i]:.3f}  {labels[i]}  {images[i].name[-30:]}")

In [ ]:
meta = spe.embed_with_meta(images[:1], model="bioclip-2-fp16")
print(meta)

Use `bioclip-2` (fp32, 1.2 GB) for full precision; `bioclip-2-fp16` is half the size.